# MVP-2.1: Pareto Diversity Correction for NSGA-II Truck-Drone Optimization

This corrected notebook preserves the MVP-2 canonical synthetic dataset while addressing the methodological weakness that NSGA-II converged toward nearly identical high-drone-adoption policies. MVP-2.1 modifies chromosome repair, decoder penalties, risk aggregation, resilience scoring, initial sampling diversity, deduplication, and representative policy selection.

The goal is not to overwrite MVP-2, but to create a corrected research-grade extension that produces interpretable trade-offs among low-carbon performance, efficiency, safety, cost, and resilience.

In [ ]:
# Cell 2: Setup

import os
import json
import math
import random
import warnings
from datetime import datetime
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt
plt.switch_backend("Agg")

BASE_DIR = Path(r"D:\02) Project Book, Paper\02) Writing Paper\05) Digital Twin-Enabled\Simulation_A_Digital_Twin_Enabled_Multi-Objective")
DATA_DIR = BASE_DIR / "data"
TABLE_DIR = BASE_DIR / "outputs" / "tables"
FIGURE_DIR = BASE_DIR / "outputs" / "figures"
PARETO_DIR = BASE_DIR / "outputs" / "pareto"
LOG_DIR = BASE_DIR / "outputs" / "logs"

for directory in [BASE_DIR, DATA_DIR, TABLE_DIR, FIGURE_DIR, PARETO_DIR, LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
rng = np.random.default_rng(RANDOM_SEED)

NOTEBOOK_FILENAME = "Simulation_A_Digital_Twin_Enabled_Multi_Objective_Truck_Drone_Logistics_MVP2_1_Pareto_Diversity_Correction.ipynb"
NOTEBOOK_PATH = BASE_DIR / NOTEBOOK_FILENAME

CANONICAL_EXPERIMENT = {
    "n_customers": 42,
    "n_depots": 1,
    "n_micro_hubs": 3,
    "n_scenarios": 24,
    "n_feasible_sorties": 71,
    "source": "advanced_synthetic_mvp1",
}
OBJECTIVE_COLS = ["duration_min", "co2_kg", "total_risk", "cost_usd", "resilience_penalty"]
MAX_DRONE_SORTIES = 18

print("MVP-2.1 paths")
for name, path in {
    "BASE_DIR": BASE_DIR,
    "DATA_DIR": DATA_DIR,
    "TABLE_DIR": TABLE_DIR,
    "FIGURE_DIR": FIGURE_DIR,
    "PARETO_DIR": PARETO_DIR,
    "LOG_DIR": LOG_DIR,
    "NOTEBOOK_PATH": NOTEBOOK_PATH,
}.items():
    print(f"{name}: {path}")
print("RANDOM_SEED:", RANDOM_SEED)

In [ ]:
# Cell 3: Load canonical MVP-1/MVP-2 inputs

def safe_read_csv(path, required=False):
    path = Path(path)
    if path.exists():
        df = pd.read_csv(path)
        print(f"FOUND: {path.name} | shape={df.shape} | columns={list(df.columns)}")
        return df
    print(f"MISSING: {path.name}")
    if required:
        raise FileNotFoundError(path)
    return None


instance_summary_df = safe_read_csv(TABLE_DIR / "table_1_synthetic_instance_summary.csv", required=True)
baseline_performance_df = safe_read_csv(TABLE_DIR / "table_2_baseline_performance.csv", required=True)
mvp1_representatives_df = safe_read_csv(TABLE_DIR / "table_3_pareto_representatives.csv", required=True)
mvp1_samples_df = safe_read_csv(TABLE_DIR / "table_solution_samples_nominal.csv", required=True)
mvp1_pareto_df = safe_read_csv(TABLE_DIR / "table_pareto_solutions_nominal.csv", required=True)
scenario_eval_mvp1_df = safe_read_csv(TABLE_DIR / "table_scenario_evaluation_representatives.csv", required=True)
scenario_robustness_mvp1_df = safe_read_csv(TABLE_DIR / "table_scenario_robustness_summary.csv", required=True)
advanced_sorties_raw = safe_read_csv(DATA_DIR / "synthetic_feasible_drone_sorties.csv", required=True)
advanced_scenarios_raw = safe_read_csv(DATA_DIR / "synthetic_scenarios.csv", required=True)

metric_lookup = dict(zip(instance_summary_df["metric"].astype(str), instance_summary_df["value"]))
for metric, expected in {
    "customers": CANONICAL_EXPERIMENT["n_customers"],
    "depots": CANONICAL_EXPERIMENT["n_depots"],
    "micro_hubs": CANONICAL_EXPERIMENT["n_micro_hubs"],
    "scenarios": CANONICAL_EXPERIMENT["n_scenarios"],
    "feasible_drone_sorties": CANONICAL_EXPERIMENT["n_feasible_sorties"],
}.items():
    observed = int(float(metric_lookup.get(metric, expected)))
    print(f"{metric}: observed={observed}, canonical={expected}")
    if observed != expected:
        warnings.warn(f"{metric} differs from canonical value. Continuing with canonical MVP-2.1 definition.")

assert len(advanced_sorties_raw) == CANONICAL_EXPERIMENT["n_feasible_sorties"]
assert len(advanced_scenarios_raw) >= CANONICAL_EXPERIMENT["n_scenarios"]

In [ ]:
# Cell 4: Corrected risk model and baseline calibration

def road_risk(distance_km, road_risk_factor=0.0008, traffic_factor=1.0):
    return float(max(distance_km, 0.0) * road_risk_factor * max(traffic_factor, 0.0))


def aerial_risk(drone_risk_score, risk_factor=1.0):
    return float(max(drone_risk_score, 0.0) * max(risk_factor, 0.0))


def total_system_risk(road_distance_km, drone_risk_score, traffic_factor=1.0, risk_factor=1.0):
    return road_risk(road_distance_km, traffic_factor=traffic_factor) + aerial_risk(drone_risk_score, risk_factor=risk_factor)


def _baseline_row(pattern):
    row = baseline_performance_df.loc[baseline_performance_df["solution_id"].astype(str).str.contains(pattern, case=False, na=False)]
    return row.iloc[0] if not row.empty else pd.Series(dtype=object)


truck_row = _baseline_row("truck")
greedy_row = _baseline_row("greedy")
truck_only_reference = {
    "policy_id": "B0_truck_only",
    "policy_label": "Truck-only baseline",
    "duration_min": float(truck_row.get("duration_min", 449.86)),
    "co2_kg": float(truck_row.get("carbon_kg", 19.88)),
    "cost_usd": float(truck_row.get("operating_cost_usd", 1345.88)),
    "truck_distance_km": float(truck_row.get("truck_distance_km", 60.24)),
    "drone_customers": 0,
    "aerial_risk_component": 0.0,
    "resilience_penalty": float(truck_row.get("resilience_penalty", 1579.10)),
}
truck_only_reference["total_risk"] = road_risk(truck_only_reference["truck_distance_km"])

greedy_reference = {
    "policy_id": "B1_greedy_truck_drone",
    "policy_label": "Greedy truck-drone baseline",
    "duration_min": float(greedy_row.get("duration_min", 359.48)),
    "co2_kg": float(greedy_row.get("carbon_kg", 16.76)),
    "cost_usd": float(greedy_row.get("operating_cost_usd", 826.93)),
    "truck_distance_km": float(greedy_row.get("truck_distance_km", 44.95)),
    "drone_customers": int(greedy_row.get("drone_customers", 18)),
    "aerial_risk_component": float(greedy_row.get("risk_score", 2.89)),
    "resilience_penalty": float(greedy_row.get("resilience_penalty", 917.51)),
}
greedy_reference["total_risk"] = road_risk(greedy_reference["truck_distance_km"]) + aerial_risk(greedy_reference["aerial_risk_component"])

risk_model_df = pd.DataFrame([
    {"component": "road_risk", "formula": "distance_km * 0.0008 * traffic_factor", "purpose": "positive truck and road exposure"},
    {"component": "aerial_risk", "formula": "drone_risk_score * risk_factor", "purpose": "aerial exposure under scenario risk"},
    {"component": "total_system_risk", "formula": "road_risk + amplified aerial risk + optional crowd penalty", "purpose": "corrected multimodal risk objective"},
])
risk_model_path = TABLE_DIR / "mvp2_1_risk_model_definition.csv"
risk_model_df.to_csv(risk_model_path, index=False)

baseline_calibration_df = pd.DataFrame([truck_only_reference, greedy_reference])
baseline_calibration_path = TABLE_DIR / "mvp2_1_baseline_calibration.csv"
baseline_calibration_df.to_csv(baseline_calibration_path, index=False)
print(baseline_calibration_df)
assert truck_only_reference["total_risk"] > 0

In [ ]:
# Cell 5: Calibrated feasible sortie table

sorties_df = pd.DataFrame({
    "sortie_id": advanced_sorties_raw["sortie_id"].astype(str),
    "customer_id": advanced_sorties_raw["customer_id"].astype(str),
    "distance_km": advanced_sorties_raw["flight_km"].astype(float),
    "duration_min": advanced_sorties_raw["base_time_min"].astype(float),
    "co2_kg": advanced_sorties_raw["base_carbon_kg"].astype(float),
    "aerial_risk": advanced_sorties_raw["base_risk_score"].astype(float),
    "cost_usd": advanced_sorties_raw["base_cost_usd"].astype(float),
    "resilience_effect": -0.35 * advanced_sorties_raw["base_time_min"].astype(float) + 2.5 * advanced_sorties_raw["base_risk_score"].astype(float),
})
sorties_df = sorties_df.sort_values(["customer_id", "duration_min", "aerial_risk"]).reset_index(drop=True)
sorties_path = TABLE_DIR / "mvp2_1_calibrated_feasible_sorties.csv"
sorties_df.to_csv(sorties_path, index=False)
print(sorties_df.describe(include="all"))
assert len(sorties_df) == CANONICAL_EXPERIMENT["n_feasible_sorties"]

In [ ]:
# Cell 6: Diversity-preserving chromosome repair and initial sampling

sortie_customer = sorties_df["customer_id"].to_numpy(str)
sortie_duration = sorties_df["duration_min"].to_numpy(float)
sortie_co2 = sorties_df["co2_kg"].to_numpy(float)
sortie_risk = sorties_df["aerial_risk"].to_numpy(float)
sortie_cost = sorties_df["cost_usd"].to_numpy(float)
sortie_ids = sorties_df["sortie_id"].to_numpy(str)

sortie_preference = (
    pd.Series(sortie_duration).rank(method="first").to_numpy()
    + pd.Series(sortie_co2).rank(method="first").to_numpy()
    + 2.0 * pd.Series(sortie_risk).rank(method="first").to_numpy()
    + pd.Series(sortie_cost).rank(method="first").to_numpy()
)


def repair_binary_selection(binary_x, sorties_df, max_drone_sorties=MAX_DRONE_SORTIES):
    binary = (np.asarray(binary_x, dtype=int) >= 1).astype(int)
    selected_indices = np.where(binary == 1)[0].tolist()
    if not selected_indices:
        return np.zeros_like(binary)
    ordered = sorted(selected_indices, key=lambda idx: sortie_preference[idx])
    seen_customers = set()
    keep = []
    for idx in ordered:
        customer = sortie_customer[idx]
        if customer in seen_customers:
            continue
        seen_customers.add(customer)
        keep.append(idx)
        if len(keep) >= int(max_drone_sorties):
            break
    repaired = np.zeros_like(binary)
    repaired[keep] = 1
    return repaired


def initialize_random_chromosome(n_genes, p=0.2, seed=None):
    local_rng = np.random.default_rng(RANDOM_SEED if seed is None else seed)
    return (local_rng.random(int(n_genes)) < float(p)).astype(int)


def make_target_k_chromosome(target_k, seed):
    local_rng = np.random.default_rng(seed)
    candidate_order = local_rng.permutation(len(sorties_df))
    x = np.zeros(len(sorties_df), dtype=int)
    seen = set()
    for idx in candidate_order:
        customer = sortie_customer[idx]
        if customer in seen:
            continue
        x[idx] = 1
        seen.add(customer)
        if int(x.sum()) >= int(target_k):
            break
    return x


def chromosome_to_selected_sorties(x, sorties_df):
    repaired = repair_binary_selection((np.asarray(x, dtype=float) >= 0.5).astype(int), sorties_df)
    return sorties_df.loc[repaired == 1].copy()


example_sparse = make_target_k_chromosome(5, RANDOM_SEED)
example_dense = make_target_k_chromosome(18, RANDOM_SEED + 1)
print("Sparse after repair:", int(repair_binary_selection(example_sparse, sorties_df).sum()))
print("Dense after repair:", int(repair_binary_selection(example_dense, sorties_df).sum()))
assert repair_binary_selection(example_sparse, sorties_df).sum() == 5
assert repair_binary_selection(example_dense, sorties_df).sum() <= 18

In [ ]:
# Cell 7: Corrected decoder and objective evaluation

def decode_policy(x, policy_id="decoded_policy"):
    repaired = repair_binary_selection((np.asarray(x, dtype=float) >= 0.5).astype(int), sorties_df)
    selected_idx = np.where(repaired == 1)[0]
    drone_customers = int(len(np.unique(sortie_customer[selected_idx]))) if len(selected_idx) else 0

    truck_only_distance = float(truck_only_reference["truck_distance_km"])
    truck_only_duration = float(truck_only_reference["duration_min"])
    truck_only_co2 = float(truck_only_reference["co2_kg"])
    truck_only_cost = float(truck_only_reference["cost_usd"])
    truck_only_resilience = float(truck_only_reference["resilience_penalty"])
    truck_only_risk = float(truck_only_reference["total_risk"])

    truck_distance_km = truck_only_distance * max(0.42, 1.0 - 0.021 * drone_customers)
    truck_distance_ratio = truck_distance_km / max(truck_only_distance, 1e-9)
    selected_duration_avg = float(sortie_duration[selected_idx].mean()) if len(selected_idx) else 0.0
    selected_co2 = float(sortie_co2[selected_idx].sum()) if len(selected_idx) else 0.0
    selected_aerial_risk = float(sortie_risk[selected_idx].sum()) if len(selected_idx) else 0.0
    selected_cost = float(sortie_cost[selected_idx].sum()) if len(selected_idx) else 0.0

    coordination_delay = 0.9 * (drone_customers ** 1.25)
    airspace_penalty = max(0, drone_customers - 10) ** 2 * 3.0
    fixed_sortie_cost = 12.0 * drone_customers
    aerial_risk_total = selected_aerial_risk * (1.0 + 0.06 * drone_customers)
    crowd_penalty = max(0, drone_customers - 10) * 0.035
    overuse_penalty = max(0, drone_customers - 12) ** 2 * 45.0

    truck_duration = truck_only_duration * truck_distance_ratio
    duration_min = max(
        0.68 * truck_only_duration,
        truck_duration + 0.70 * selected_duration_avg + coordination_delay + airspace_penalty,
    )

    co2_kg = truck_only_co2 * truck_distance_ratio + selected_co2 + 0.015 * drone_customers
    cost_usd = truck_only_cost * truck_distance_ratio + selected_cost + fixed_sortie_cost + 2.0 * airspace_penalty
    total_risk = road_risk(truck_distance_km) + aerial_risk_total + crowd_penalty

    duration_ratio = duration_min / max(truck_only_duration, 1e-9)
    cost_ratio = cost_usd / max(truck_only_cost, 1e-9)
    risk_ratio = total_risk / max(truck_only_risk, 1e-9)
    resilience_penalty = (
        truck_only_resilience
        * (0.42 * duration_ratio + 0.24 * cost_ratio + 0.24 * min(risk_ratio, 30.0) / 30.0 + 0.10)
        + overuse_penalty
        + 4.0 * airspace_penalty
    )

    return {
        "policy_id": policy_id,
        "drone_customers": drone_customers,
        "n_selected_sorties": int(repaired.sum()),
        "truck_distance_km": float(truck_distance_km),
        "duration_min": float(duration_min),
        "co2_kg": float(co2_kg),
        "total_risk": float(total_risk),
        "cost_usd": float(cost_usd),
        "resilience_penalty": float(resilience_penalty),
        "coordination_delay": float(coordination_delay),
        "airspace_penalty": float(airspace_penalty),
        "overuse_penalty": float(overuse_penalty),
        "selected_sortie_ids": "|".join(sortie_ids[selected_idx].tolist()) if len(selected_idx) else "",
    }


def evaluate_chromosome(x):
    decoded = decode_policy(x)
    return np.array([decoded[col] for col in OBJECTIVE_COLS], dtype=float)


sanity_records = []
for k in [0, 3, 5, 8, 10, 12, 14, 16, 18]:
    decoded = decode_policy(make_target_k_chromosome(k, RANDOM_SEED + k), policy_id=f"k_{k:02d}")
    sanity_records.append(decoded)
sanity_df = pd.DataFrame(sanity_records)
print(sanity_df[["policy_id", "drone_customers"] + OBJECTIVE_COLS + ["airspace_penalty", "overuse_penalty"]])
assert np.isfinite(sanity_df[OBJECTIVE_COLS].to_numpy(float)).all()
assert (sanity_df[OBJECTIVE_COLS].to_numpy(float) >= 0).all()
assert sanity_df.loc[sanity_df["drone_customers"].idxmax(), "total_risk"] > sanity_df.loc[sanity_df["drone_customers"].idxmin(), "total_risk"]

In [ ]:
# Cell 8: pymoo availability and diversity sampling

try:
    import pymoo
    from pymoo.core.problem import ElementwiseProblem
    from pymoo.core.sampling import Sampling
    from pymoo.algorithms.moo.nsga2 import NSGA2
    from pymoo.optimize import minimize
    print("pymoo version:", getattr(pymoo, "__version__", "unknown"))
except ModuleNotFoundError as exc:
    print("pymoo is not installed. Run: pip install pymoo")
    raise ImportError("MVP-2.1 requires pymoo for NSGA-II diversity-corrected optimization.") from exc


class DroneDiversitySampling(Sampling):
    def _do(self, problem, n_samples, **kwargs):
        local_rng = np.random.default_rng(RANDOM_SEED)
        bands = [(0, 5), (6, 10), (11, 14), (15, 18)]
        X = np.zeros((n_samples, problem.n_var), dtype=float)
        for i in range(n_samples):
            low, high = bands[i % len(bands)]
            target_k = int(local_rng.integers(low, high + 1))
            binary = make_target_k_chromosome(target_k, RANDOM_SEED + 1000 + i)
            repaired = repair_binary_selection(binary, sorties_df)
            X[i, :] = np.where(repaired == 1, local_rng.uniform(0.62, 1.0, problem.n_var), local_rng.uniform(0.0, 0.38, problem.n_var))
        return X

In [ ]:
# Cell 9: Define diversity-corrected NSGA-II problem

class TruckDroneDiversityProblem(ElementwiseProblem):
    def __init__(self):
        super().__init__(n_var=len(sorties_df), n_obj=len(OBJECTIVE_COLS), n_constr=0, xl=0.0, xu=1.0)

    def _evaluate(self, x, out, *args, **kwargs):
        repaired = repair_binary_selection((np.asarray(x, dtype=float) >= 0.5).astype(int), sorties_df)
        out["F"] = evaluate_chromosome(repaired)


problem = TruckDroneDiversityProblem()
test_out = {}
problem._evaluate(np.random.default_rng(RANDOM_SEED).random(problem.n_var), test_out)
print("Test F:", test_out["F"])
assert test_out["F"].shape == (len(OBJECTIVE_COLS),)

In [ ]:
# Cell 10: Run NSGA-II with diversity-corrected decoder

import time

ESTIMATED_RUNTIME_HIGH = True
if ESTIMATED_RUNTIME_HIGH:
    population_size = 120
    generations = 220
    reduced_settings_used = True
else:
    population_size = 160
    generations = 300
    reduced_settings_used = False

nsga2_settings = {
    "population_size": population_size,
    "generations": generations,
    "seed": RANDOM_SEED,
    "reduced_settings_used": reduced_settings_used,
    "diversity_sampling": "DroneDiversitySampling with 0-5, 6-10, 11-14, 15-18 drone-sortie bands",
}

algorithm = NSGA2(pop_size=population_size, sampling=DroneDiversitySampling())
start = time.perf_counter()
result = minimize(problem, algorithm, ("n_gen", generations), seed=RANDOM_SEED, verbose=True)
nsga2_settings["runtime_seconds"] = float(time.perf_counter() - start)

final_X = result.pop.get("X")
if final_X.ndim == 1:
    final_X = final_X.reshape(1, -1)

raw_records = []
for idx, x in enumerate(final_X):
    repaired = repair_binary_selection((np.asarray(x) >= 0.5).astype(int), sorties_df)
    decoded = decode_policy(repaired, policy_id=f"MVP2_1_NSGA2_{idx:04d}")
    raw_records.append(decoded)

raw_population_df = pd.DataFrame(raw_records)
raw_population_path = PARETO_DIR / "mvp2_1_nsga2_final_population_raw.csv"
raw_population_df.to_csv(raw_population_path, index=False)
print("NSGA-II settings:", nsga2_settings)
print("Raw final population drone customer values:", sorted(raw_population_df["drone_customers"].unique().tolist()))
print("Saved raw population:", raw_population_path)

In [ ]:
# Cell 11: Deduplicate final population and extract Pareto front

def is_dominated(a, b, objective_cols):
    return all(float(b[col]) <= float(a[col]) for col in objective_cols) and any(float(b[col]) < float(a[col]) for col in objective_cols)


def extract_pareto_front(df, objective_cols):
    records = df.to_dict("records")
    keep = []
    for i, candidate in enumerate(records):
        keep.append(not any(i != j and is_dominated(candidate, challenger, objective_cols) for j, challenger in enumerate(records)))
    return df.loc[keep].sort_values(objective_cols).reset_index(drop=True)


rounded_cols = [f"{col}_round6" for col in OBJECTIVE_COLS]
dedup_working = raw_population_df.copy()
for col, rounded_col in zip(OBJECTIVE_COLS, rounded_cols):
    dedup_working[rounded_col] = dedup_working[col].round(6)
deduplicated_population_df = (
    dedup_working
    .sort_values(["drone_customers"] + OBJECTIVE_COLS)
    .drop_duplicates(subset=rounded_cols, keep="first")
    .drop(columns=rounded_cols)
    .reset_index(drop=True)
)
pareto_front_df = extract_pareto_front(deduplicated_population_df, OBJECTIVE_COLS)

dedup_path = PARETO_DIR / "mvp2_1_nsga2_final_population_deduplicated.csv"
pareto_path = PARETO_DIR / "mvp2_1_nsga2_pareto_front.csv"
deduplicated_population_df.to_csv(dedup_path, index=False)
pareto_front_df.to_csv(pareto_path, index=False)

print("Raw final policies:", len(raw_population_df))
print("Deduplicated policies:", len(deduplicated_population_df))
print("Pareto policies:", len(pareto_front_df))
print("Pareto drone customer values:", sorted(pareto_front_df["drone_customers"].unique().tolist()))
assert not pareto_front_df.empty

In [ ]:
# Cell 12: Representative policies and diversity diagnostics

def normalized_score(df, weights):
    scores = np.zeros(len(df), dtype=float)
    for col, weight in weights.items():
        values = df[col].astype(float)
        denom = max(float(values.max() - values.min()), 1e-12)
        scores += float(weight) * ((values - values.min()) / denom).to_numpy()
    return scores


weights = {
    "balanced_compromise": {"duration_min": 0.20, "co2_kg": 0.20, "total_risk": 0.25, "cost_usd": 0.15, "resilience_penalty": 0.20},
    "safety_first": {"duration_min": 0.10, "co2_kg": 0.10, "total_risk": 0.58, "cost_usd": 0.07, "resilience_penalty": 0.15},
    "low_carbon": {"duration_min": 0.12, "co2_kg": 0.58, "total_risk": 0.13, "cost_usd": 0.07, "resilience_penalty": 0.10},
    "efficiency_first": {"duration_min": 0.58, "co2_kg": 0.12, "total_risk": 0.10, "cost_usd": 0.10, "resilience_penalty": 0.10},
}

representative_specs = {
    "min_duration": ("duration_min", None),
    "min_co2": ("co2_kg", None),
    "min_risk": ("total_risk", None),
    "min_cost": ("cost_usd", None),
    "min_resilience_penalty": ("resilience_penalty", None),
    "balanced_compromise": (None, weights["balanced_compromise"]),
    "safety_first": (None, weights["safety_first"]),
    "low_carbon": (None, weights["low_carbon"]),
    "efficiency_first": (None, weights["efficiency_first"]),
}

representative_records = []
for representative_type, (selector, weight_map) in representative_specs.items():
    working = pareto_front_df.copy()
    if selector is not None:
        selected_idx = working[selector].astype(float).idxmin()
        selection_score = 0.0
    else:
        scores = normalized_score(working, weight_map)
        selected_idx = working.index[int(np.argmin(scores))]
        selection_score = float(scores[int(np.argmin(scores))])
    row = working.loc[selected_idx].copy()
    row["representative_type"] = representative_type
    row["selection_score"] = selection_score
    representative_records.append(row)

representative_policies_df = pd.DataFrame(representative_records)
representative_path = TABLE_DIR / "mvp2_1_representative_policies.csv"
representative_policies_df.to_csv(representative_path, index=False)

key_types = ["balanced_compromise", "low_carbon", "safety_first", "efficiency_first"]
key_policy_ids = representative_policies_df.loc[representative_policies_df["representative_type"].isin(key_types), "policy_id"].astype(str).tolist()
distinct_key_representatives = len(set(key_policy_ids)) > 1

diversity_records = [
    {"metric": "raw_final_policies", "value": len(raw_population_df)},
    {"metric": "unique_final_policies", "value": len(deduplicated_population_df)},
    {"metric": "pareto_policies", "value": len(pareto_front_df)},
    {"metric": "unique_drone_customers", "value": ",".join(map(str, sorted(pareto_front_df["drone_customers"].unique().tolist())))},
    {"metric": "min_drone_customers", "value": int(pareto_front_df["drone_customers"].min())},
    {"metric": "max_drone_customers", "value": int(pareto_front_df["drone_customers"].max())},
    {"metric": "key_representatives_distinct", "value": bool(distinct_key_representatives)},
]
for col in OBJECTIVE_COLS:
    diversity_records.append({"metric": f"{col}_range", "value": float(pareto_front_df[col].max() - pareto_front_df[col].min())})
diversity_diagnostics_df = pd.DataFrame(diversity_records)
diagnostics_path = TABLE_DIR / "mvp2_1_pareto_diversity_diagnostics.csv"
diversity_diagnostics_df.to_csv(diagnostics_path, index=False)

print(representative_policies_df[["representative_type", "policy_id", "drone_customers"] + OBJECTIVE_COLS])
print(diversity_diagnostics_df)

In [ ]:
# Cell 13: Baseline, MVP-1, and NSGA-II comparison

def improvement(reference, value):
    return 0.0 if abs(float(reference)) < 1e-12 else float(100.0 * (float(reference) - float(value)) / abs(float(reference)))


comparison_records = []
for reference in [truck_only_reference, greedy_reference]:
    comparison_records.append({
        "policy_type": reference["policy_label"],
        "policy_id": reference["policy_id"],
        "drone_customers": int(reference["drone_customers"]),
        "truck_distance_km": float(reference["truck_distance_km"]),
        **{col: float(reference[col]) for col in OBJECTIVE_COLS},
    })

p0229 = mvp1_pareto_df.loc[mvp1_pareto_df["solution_id"].astype(str).eq("P0229")]
if not p0229.empty:
    row = p0229.iloc[0]
    p0229_distance = float(row.get("truck_distance_km", truck_only_reference["truck_distance_km"] * 0.70))
    p0229_risk_score = float(row.get("risk_score", 2.086))
    comparison_records.append({
        "policy_type": "MVP-1 sampled P0229",
        "policy_id": "P0229",
        "drone_customers": int(row.get("drone_customers", 9)),
        "truck_distance_km": p0229_distance,
        "duration_min": float(row.get("duration_min", 383.76)),
        "co2_kg": float(row.get("carbon_kg", 14.97)),
        "total_risk": total_system_risk(p0229_distance, p0229_risk_score),
        "cost_usd": float(row.get("operating_cost_usd", 911.06)),
        "resilience_penalty": float(row.get("resilience_penalty", 1024.55)),
    })

type_map = {
    "balanced_compromise": "NSGA-II balanced",
    "low_carbon": "NSGA-II low-carbon",
    "safety_first": "NSGA-II safety-first",
    "efficiency_first": "NSGA-II efficiency-first",
}
for representative_type, policy_type in type_map.items():
    row = representative_policies_df.loc[representative_policies_df["representative_type"].eq(representative_type)].iloc[0]
    comparison_records.append({
        "policy_type": policy_type,
        "policy_id": row["policy_id"],
        "drone_customers": int(row["drone_customers"]),
        "truck_distance_km": float(row["truck_distance_km"]),
        **{col: float(row[col]) for col in OBJECTIVE_COLS},
    })

comparison_df = pd.DataFrame(comparison_records)
truck_ref = comparison_df.loc[comparison_df["policy_id"].eq("B0_truck_only")].iloc[0]
for col in ["duration_min", "co2_kg", "cost_usd", "resilience_penalty"]:
    comparison_df[f"{col}_improvement_percent"] = comparison_df[col].apply(lambda value: improvement(truck_ref[col], value))
comparison_df["risk_change_percent"] = comparison_df["total_risk"].apply(
    lambda value: 0.0 if abs(float(truck_ref["total_risk"])) < 1e-12 else 100.0 * (float(value) - float(truck_ref["total_risk"])) / abs(float(truck_ref["total_risk"]))
)
comparison_path = TABLE_DIR / "mvp2_1_baseline_nsga2_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)
print(comparison_df)

In [ ]:
# Cell 14: Scenario robustness under corrected risk and resilience model

def build_scenarios():
    scenarios = advanced_scenarios_raw.head(24).copy()
    scenarios["demand_factor"] = scenarios.get("demand_multiplier", 1.0).astype(float)
    scenarios["traffic_factor"] = 1.0 + scenarios.get("congestion_index", 0.35).astype(float) + 0.35 * scenarios.get("road_disruption_index", 0.0).astype(float)
    scenarios["wind_factor"] = 1.0 + scenarios.get("wind_speed_mps", 3.0).astype(float) / 30.0
    scenarios["risk_factor"] = 1.0 + 1.8 * scenarios.get("aerial_risk_alert", 0.1).astype(float)
    scenarios["airspace_capacity_factor"] = np.clip(1.0 - 0.35 * scenarios.get("aerial_risk_alert", 0.1).astype(float), 0.55, 1.0)
    return scenarios[["scenario_id", "scenario_label", "demand_factor", "traffic_factor", "wind_factor", "risk_factor", "airspace_capacity_factor"]]


scenario_df = build_scenarios()
scenario_records = []
for _, scenario in scenario_df.iterrows():
    capacity_stress = max(0.0, 1.0 - float(scenario["airspace_capacity_factor"]))
    for _, policy in comparison_df.iterrows():
        drone_share = float(policy["drone_customers"]) / CANONICAL_EXPERIMENT["n_customers"]
        duration = float(policy["duration_min"]) * (
            1.0
            + 0.30 * (float(scenario["demand_factor"]) - 1.0)
            + 0.26 * (float(scenario["traffic_factor"]) - 1.0) * (1.0 - drone_share)
            + 0.18 * (float(scenario["wind_factor"]) - 1.0) * drone_share
            + 0.55 * capacity_stress * drone_share
        )
        co2 = float(policy["co2_kg"]) * (
            1.0
            + 0.10 * (float(scenario["traffic_factor"]) - 1.0) * (1.0 - drone_share)
            + 0.18 * (float(scenario["wind_factor"]) - 1.0) * drone_share
        )
        total_risk_value = float(policy["total_risk"]) * (
            1.0
            + 0.32 * (float(scenario["traffic_factor"]) - 1.0) * (1.0 - drone_share)
            + 0.70 * (float(scenario["risk_factor"]) - 1.0) * max(drone_share, 0.05)
            + 0.40 * capacity_stress * drone_share
        )
        cost = float(policy["cost_usd"]) * (
            1.0
            + 0.15 * (float(scenario["demand_factor"]) - 1.0)
            + 0.12 * (float(scenario["traffic_factor"]) - 1.0)
            + 0.25 * capacity_stress * drone_share
        )
        resilience = float(policy["resilience_penalty"]) * (
            1.0
            + 0.25 * (float(scenario["traffic_factor"]) - 1.0)
            + 0.28 * (float(scenario["risk_factor"]) - 1.0)
            + 0.20 * (float(scenario["demand_factor"]) - 1.0)
            + 0.65 * capacity_stress * drone_share
        )
        scenario_records.append({
            "scenario_id": scenario["scenario_id"],
            "scenario_label": scenario["scenario_label"],
            "policy_type": policy["policy_type"],
            "policy_id": policy["policy_id"],
            "duration_min": duration,
            "co2_kg": co2,
            "total_risk": total_risk_value,
            "cost_usd": cost,
            "resilience_penalty": resilience,
            "service_level": 1.0,
        })

scenario_evaluation_df = pd.DataFrame(scenario_records)
scenario_robustness_summary_df = scenario_evaluation_df.groupby(["policy_type", "policy_id"], as_index=False).agg(
    mean_duration_min=("duration_min", "mean"),
    p90_duration_min=("duration_min", lambda s: float(np.percentile(s, 90))),
    mean_co2_kg=("co2_kg", "mean"),
    p90_co2_kg=("co2_kg", lambda s: float(np.percentile(s, 90))),
    mean_total_risk=("total_risk", "mean"),
    p90_total_risk=("total_risk", lambda s: float(np.percentile(s, 90))),
    mean_cost_usd=("cost_usd", "mean"),
    p90_cost_usd=("cost_usd", lambda s: float(np.percentile(s, 90))),
    mean_resilience_penalty=("resilience_penalty", "mean"),
    p90_resilience_penalty=("resilience_penalty", lambda s: float(np.percentile(s, 90))),
)
scenario_eval_path = TABLE_DIR / "mvp2_1_nsga2_scenario_evaluation.csv"
scenario_summary_path = TABLE_DIR / "mvp2_1_scenario_robustness_summary.csv"
scenario_evaluation_df.to_csv(scenario_eval_path, index=False)
scenario_robustness_summary_df.to_csv(scenario_summary_path, index=False)
print(scenario_robustness_summary_df)

In [ ]:
# Cell 15: Manuscript-ready diversity figures

figure_paths = []

fig, ax = plt.subplots(figsize=(7.2, 5.2))
ax.scatter(deduplicated_population_df["duration_min"], deduplicated_population_df["co2_kg"], s=20, alpha=0.35, label="Deduplicated final population")
ax.scatter(pareto_front_df["duration_min"], pareto_front_df["co2_kg"], s=38, color="crimson", label="Pareto front")
ax.set_xlabel("Duration (min)")
ax.set_ylabel("CO2 emissions (kg)")
ax.set_title("MVP-2.1 Pareto diversity: duration vs CO2")
ax.legend()
fig.tight_layout()
path = FIGURE_DIR / "mvp2_1_pareto_duration_co2.png"
fig.savefig(path, dpi=300)
plt.close(fig)
figure_paths.append(path)

fig, ax = plt.subplots(figsize=(7.2, 5.2))
ax.scatter(deduplicated_population_df["co2_kg"], deduplicated_population_df["total_risk"], s=20, alpha=0.35, label="Deduplicated final population")
ax.scatter(pareto_front_df["co2_kg"], pareto_front_df["total_risk"], s=38, color="crimson", label="Pareto front")
ax.set_xlabel("CO2 emissions (kg)")
ax.set_ylabel("Corrected total system risk")
ax.set_title("MVP-2.1 Pareto diversity: CO2 vs risk")
ax.legend()
fig.tight_layout()
path = FIGURE_DIR / "mvp2_1_pareto_co2_risk.png"
fig.savefig(path, dpi=300)
plt.close(fig)
figure_paths.append(path)

fig, ax = plt.subplots(figsize=(7.4, 5.2))
scatter = ax.scatter(pareto_front_df["drone_customers"], pareto_front_df["co2_kg"], c=pareto_front_df["total_risk"], cmap="viridis", s=44)
ax.set_xlabel("Drone-served customers")
ax.set_ylabel("CO2 emissions (kg)")
ax.set_title("MVP-2.1 adoption trade-off: drone use, carbon, and risk")
cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label("Corrected total risk")
fig.tight_layout()
path = FIGURE_DIR / "mvp2_1_pareto_drone_customers_tradeoff.png"
fig.savefig(path, dpi=300)
plt.close(fig)
figure_paths.append(path)

comparison_plot = comparison_df.set_index("policy_type")[OBJECTIVE_COLS].copy()
comparison_plot = comparison_plot / comparison_plot.max(axis=0).replace(0, 1)
fig, ax = plt.subplots(figsize=(10.5, 5.8))
comparison_plot.plot(kind="bar", ax=ax)
ax.set_ylabel("Normalized objective value")
ax.set_xlabel("")
ax.set_title("MVP-2.1 baseline and representative policy comparison")
ax.tick_params(axis="x", rotation=30)
ax.legend(ncol=2, fontsize=8)
fig.tight_layout()
path = FIGURE_DIR / "mvp2_1_baseline_vs_nsga2_objectives.png"
fig.savefig(path, dpi=300)
plt.close(fig)
figure_paths.append(path)

fig, ax = plt.subplots(figsize=(9.0, 5.2))
for policy_type, group in scenario_evaluation_df.groupby("policy_type"):
    if policy_type in ["Truck-only baseline", "Greedy truck-drone baseline", "NSGA-II balanced", "NSGA-II low-carbon", "NSGA-II safety-first", "NSGA-II efficiency-first"]:
        ordered = group.sort_values("scenario_id").reset_index(drop=True)
        ax.plot(range(len(ordered)), ordered["resilience_penalty"], marker="o", markersize=2.8, linewidth=1.2, label=policy_type)
ax.set_xlabel("Scenario index")
ax.set_ylabel("Resilience penalty")
ax.set_title("MVP-2.1 scenario robustness profiles")
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
path = FIGURE_DIR / "mvp2_1_scenario_robustness_profiles.png"
fig.savefig(path, dpi=300)
plt.close(fig)
figure_paths.append(path)

for path in figure_paths:
    print(path)
    assert path.exists() and path.stat().st_size > 0

In [ ]:
# Cell 16: Success criteria, summary, and ZIP archive

critical_outputs = [
    PARETO_DIR / "mvp2_1_nsga2_final_population_raw.csv",
    PARETO_DIR / "mvp2_1_nsga2_final_population_deduplicated.csv",
    PARETO_DIR / "mvp2_1_nsga2_pareto_front.csv",
    TABLE_DIR / "mvp2_1_representative_policies.csv",
    TABLE_DIR / "mvp2_1_baseline_nsga2_comparison.csv",
    TABLE_DIR / "mvp2_1_scenario_robustness_summary.csv",
    TABLE_DIR / "mvp2_1_pareto_diversity_diagnostics.csv",
    FIGURE_DIR / "mvp2_1_pareto_duration_co2.png",
    FIGURE_DIR / "mvp2_1_pareto_co2_risk.png",
    FIGURE_DIR / "mvp2_1_pareto_drone_customers_tradeoff.png",
    FIGURE_DIR / "mvp2_1_baseline_vs_nsga2_objectives.png",
    FIGURE_DIR / "mvp2_1_scenario_robustness_profiles.png",
]

min_risk_k = int(representative_policies_df.loc[representative_policies_df["representative_type"].eq("min_risk"), "drone_customers"].iloc[0])
min_co2_k = int(representative_policies_df.loc[representative_policies_df["representative_type"].eq("min_co2"), "drone_customers"].iloc[0])
min_duration_k = int(representative_policies_df.loc[representative_policies_df["representative_type"].eq("min_duration"), "drone_customers"].iloc[0])

success_checks = {
    "pareto_at_least_15_unique_policies": len(pareto_front_df) >= 15,
    "pareto_at_least_4_drone_customer_values": pareto_front_df["drone_customers"].nunique() >= 4,
    "key_representatives_not_all_same": len(set(representative_policies_df.loc[representative_policies_df["representative_type"].isin(["balanced_compromise", "low_carbon", "safety_first", "efficiency_first"]), "policy_id"].astype(str))) > 1,
    "min_risk_fewer_drones_than_min_co2_or_min_duration": min_risk_k < max(min_co2_k, min_duration_k),
    "no_critical_output_missing": all(path.exists() and path.stat().st_size > 0 for path in critical_outputs),
}
diversity_passed = all(success_checks.values())

summary_lines = [
    "MVP-2.1 Pareto Diversity Correction Execution Summary",
    "====================================================",
    f"Notebook filename: {NOTEBOOK_FILENAME}",
    f"Execution date/time: {datetime.now().astimezone().isoformat(timespec='seconds')}",
    f"Diversity diagnostics passed: {diversity_passed}",
    f"Number of raw final policies: {len(raw_population_df)}",
    f"Number of unique final policies: {len(deduplicated_population_df)}",
    f"Number of Pareto policies: {len(pareto_front_df)}",
    f"Unique drone customer values: {sorted(pareto_front_df['drone_customers'].unique().tolist())}",
    "",
    "Selected representative policies:",
    representative_policies_df[["representative_type", "policy_id", "drone_customers"] + OBJECTIVE_COLS].to_string(index=False),
    "",
    "Success checks:",
    json.dumps(success_checks, indent=2),
    "",
    "Key improvements over MVP-2:",
    "- Repair preserves the number of non-conflicting selected sorties rather than filling up to 18.",
    "- Decoder adds nonlinear coordination delay, aerial-risk amplification, fixed sortie cost, airspace congestion, crowd exposure, and overuse resilience penalties.",
    "- Initial NSGA-II sampling includes low, moderate, and high drone-adoption bands.",
    "- Final population is deduplicated before Pareto extraction and representative selection.",
    "",
    "Remaining limitations:",
    "- Still uses the canonical synthetic network rather than OSMnx road data.",
    "- Scenario robustness is based on deterministic stress multipliers rather than a live digital twin simulator.",
    "- Drone sortie feasibility is inherited from the advanced synthetic MVP-1 dataset.",
    "",
    f"Recommendation whether results are manuscript-ready: {'YES for MVP-2.1 computational results, with limitations stated' if diversity_passed else 'NO; diversity criteria require further tuning'}",
]
summary_text = "\n".join(summary_lines)
summary_path = BASE_DIR / "MVP2_1_execution_summary.txt"
summary_path.write_text(summary_text, encoding="utf-8")
print(summary_text)

zip_path = BASE_DIR / "MVP2_1_Pareto_Diversity_Correction_outputs.zip"
zip_inputs = [NOTEBOOK_PATH, summary_path]
zip_inputs.extend(sorted(TABLE_DIR.glob("mvp2_1*.csv")))
zip_inputs.extend(sorted(PARETO_DIR.glob("mvp2_1*.csv")))
zip_inputs.extend(sorted(FIGURE_DIR.glob("mvp2_1*.png")))
unique_zip_inputs = []
seen = set()
for path in zip_inputs:
    if path.exists() and path not in seen:
        unique_zip_inputs.append(path)
        seen.add(path)

with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as zf:
    for path in unique_zip_inputs:
        zf.write(path, arcname=str(path.relative_to(BASE_DIR)))

print("ZIP path:", zip_path)
print("ZIP file count:", len(unique_zip_inputs))
print("FINAL STATUS:", "PASS" if diversity_passed else "FAIL")
if not diversity_passed:
    raise AssertionError(f"MVP-2.1 success criteria failed: {success_checks}")